# Lab 03 ML Pipeline (Variant 9)

**Setup -> ETL -> Features -> ML -> Eval**

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, size, coalesce, to_timestamp, element_at, hour, when, date_format, lit
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder.appName("Lab3_Variant9_Endomondo_RF").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")


## Setup

In [ ]:
candidates = ["data/endomondoHR.json", "data/endomondo_sample.json"]
input_path = next((p for p in candidates if Path(p).exists()), None)
if input_path is None:
    raise FileNotFoundError(f"No dataset found in: {candidates}")

print("Input:", input_path)


## ETL

In [ ]:
df = spark.read.json(input_path)
df.printSchema()

work = df.dropna(subset=["sport"]).filter(size(col("heart_rate")) > 0)

start_ts = coalesce(
    to_timestamp(col("timestamp_start")),
    to_timestamp(element_at(col("timestamps"), 1))
)
end_ts = coalesce(
    to_timestamp(col("timestamp_end")),
    to_timestamp(element_at(col("timestamps"), -1))
)

work = work.withColumn("start_ts", start_ts)
work = work.withColumn("duration_minutes", ((end_ts.cast("long") - start_ts.cast("long")) / 60.0))
work = work.withColumn(
    "duration_minutes",
    when(col("duration_minutes") > 0, col("duration_minutes")).otherwise(size(col("heart_rate")).cast("double"))
)

work = work.withColumn("start_hour", hour(col("start_ts")))
work = work.withColumn(
    "time_of_day",
    when(col("start_hour").between(5, 11), lit("morning"))
    .when(col("start_hour").between(12, 16), lit("day"))
    .when(col("start_hour").between(17, 22), lit("evening"))
    .otherwise(lit("night"))
)
work = work.withColumn("day_of_week", coalesce(date_format(col("start_ts"), "E"), lit("unknown")))

work.select("sport", "time_of_day", "day_of_week", "duration_minutes").show(10, truncate=False)


## Features

In [ ]:
tod_indexer = StringIndexer(inputCol="time_of_day", outputCol="time_of_day_idx", handleInvalid="keep")
dow_indexer = StringIndexer(inputCol="day_of_week", outputCol="day_of_week_idx", handleInvalid="keep")
label_indexer = StringIndexer(inputCol="sport", outputCol="label", handleInvalid="skip")

assembler = VectorAssembler(
    inputCols=["time_of_day_idx", "day_of_week_idx", "duration_minutes"],
    outputCol="features_raw"
)

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)


## ML

In [ ]:
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=80, maxDepth=8, seed=42)

pipeline = Pipeline(stages=[tod_indexer, dow_indexer, label_indexer, assembler, scaler, rf])

train_df, test_df = work.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_df)
pred = model.transform(test_df)

pred.select("sport", "label", "prediction", "probability").show(10, truncate=False)


## Eval

In [ ]:
acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

accuracy = acc_eval.evaluate(pred)
f1 = f1_eval.evaluate(pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}")

cm = pred.groupBy("label", "prediction").count().orderBy("label", "prediction")
cm.show(truncate=False)


## Business Interpretation

In [ ]:
rf_model = model.stages[-1]
importance = rf_model.featureImportances.toArray().tolist()
feature_names = ["time_of_day_idx", "day_of_week_idx", "duration_minutes"]

pairs = sorted(zip(feature_names, importance), key=lambda x: x[1], reverse=True)
print("Feature importances:")
for n, s in pairs:
    print(f"{n}: {s:.4f}")

print("\nBusiness takeaway:")
print("Temporal features (time/day) are strong predictors of activity type.")
print("Use this for push personalization: morning-running and evening-gym scenarios.")


## Finish

In [ ]:
spark.stop()
